In [27]:
!pip3 install docling
!pip3 install pillow


[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [28]:
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat

pipeline_options = PdfPipelineOptions()
pipeline_options.generate_picture_images = True
pipeline_options.images_scale = 2.0

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
    }
)

result = converter.convert("/Users/atharv.muchandi/udemy/HR Policy Manual 2023.pdf")

doc = result.document

[INFO] 2026-08-06 10:19:38,967 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-08-06 10:19:39,002 [RapidOCR] download_file.py:60: File exists and is valid: /Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-08-06 10:19:39,004 [RapidOCR] main.py:63: Using /Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-08-06 10:19:39,035 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-08-06 10:19:39,065 [RapidOCR] download_file.py:60: File exists and is valid: /Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-08-06 10:19:39,066 [RapidOCR] main.py:63: Using /Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-08-06 10:19:39

In [29]:
from docling_core.types.doc import ImageRefMode

markdown = doc.export_to_markdown(image_mode=ImageRefMode.REFERENCED)

with open("document.md", "w", encoding="utf-8") as f:
    f.write(markdown)


print(markdown)

<!-- image -->

INDIAN INSTITUTE  MANAGEMENT AHMEDABAD

## HUMAN RESOURCES POLICY MANUAL

## IIMA HR Policy Manual 2023 STAFF 2023

a

b

IIMA HR Policy Manual 2023

## DECLARATION

The  objective  of  this  Manual  is  to  compile    the  HR  policies  and procedures followed in IIMA. It also presents the general rules and regulations  that govern the employees of the Institute.

This  Manual  supersedes  all  previous  manuals,  handbooks,  and memorandums  that  may  have  been  issued  from  time  to  time  on subjects covered in this Manual.

The Institute reserves its right to interpret; change; suspend; cancel; or dispute, with or without notice; all or any part of what is contained in the Manual. The Institute will notify all employees of such changes.

In  the  interpretation  of  any  policies  and  procedures  covered  in the Manual, the Director's decision will be final and binding on all employees of the Institute.

## HR Department

ii IIMA HR Policy Manual 2023

Contents

In [30]:
import json

data = doc.export_to_dict()

with open("document.json", "w") as f:
    json.dump(data, f, indent=4)

In [31]:
print(doc.export_to_dict().keys())

dict_keys(['schema_name', 'version', 'name', 'origin', 'furniture', 'body', 'groups', 'texts', 'pictures', 'tables', 'key_value_items', 'form_items', 'pages'])


In [32]:
print(len(data["tables"]))

116


In [33]:
print(len(data["pictures"]))

16


In [34]:
!pip3 install docling-core sentence-transformers chromadb


[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [35]:
from docling_core.transforms.chunker import HybridChunker

chunker = HybridChunker(
    tokenizer="sentence-transformers/all-MiniLM-L6-v2",
    max_tokens=512,
)

chunks = list(chunker.chunk(doc))
chunk_texts = [chunker.serialize(c) for c in chunks]
chunk_metas = [
    {
        "page": (c.meta.doc_items[0].prov[0].page_no if c.meta.doc_items and c.meta.doc_items[0].prov else None),
        "headings": " > ".join(c.meta.headings) if c.meta.headings else "",
    }
    for c in chunks
]
print(f"Total chunks: {len(chunks)}")

for i, text in enumerate(chunk_texts):
    print(f"--- Chunk {i} (page={chunk_metas[i]['page']}, headings={chunk_metas[i]['headings']}) ---")
    print(text)
    print()

    


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (3617 > 512). Running this sequence through the model will result in indexing errors


Total chunks: 500
--- Chunk 0 (page=1, headings=) ---
INDIAN INSTITUTE  MANAGEMENT AHMEDABAD

--- Chunk 1 (page=1, headings=IIMA HR Policy Manual 2023 STAFF 2023) ---
IIMA HR Policy Manual 2023 STAFF 2023
a
b
IIMA HR Policy Manual 2023

--- Chunk 2 (page=3, headings=DECLARATION) ---
DECLARATION
The  objective  of  this  Manual  is  to  compile    the  HR  policies  and procedures followed in IIMA. It also presents the general rules and regulations  that govern the employees of the Institute.
This  Manual  supersedes  all  previous  manuals,  handbooks,  and memorandums  that  may  have  been  issued  from  time  to  time  on subjects covered in this Manual.
The Institute reserves its right to interpret; change; suspend; cancel; or dispute, with or without notice; all or any part of what is contained in the Manual. The Institute will notify all employees of such changes.
In  the  interpretation  of  any  policies  and  procedures  covered  in the Manual, the Director's decision will be 

/var/folders/vc/47nq4gp93_b0p5h81kh5xmtm0000gp/T/ipykernel_3737/3659713539.py:9: DeprecationWarning: Use contextualize() instead.
  chunk_texts = [chunker.serialize(c) for c in chunks]


In [36]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embed_model.encode(chunk_texts, show_progress_bar=True, batch_size=32)
print(embeddings.shape)


Batches: 100%|██████████| 16/16 [00:00<00:00, 20.75it/s]

(500, 384)


In [37]:
from pymilvus import MilvusClient

MILVUS_URI = "http://localhost:19530"
milvus_client = MilvusClient(uri=MILVUS_URI)
print(milvus_client.list_collections())


['hr_policy_manual_modern_rag', 'hr_policy_manual']


In [38]:
collection_name = "hr_policy_manual"

if milvus_client.has_collection(collection_name):
    milvus_client.drop_collection(collection_name)

milvus_client.create_collection(
    collection_name=collection_name,
    dimension=embeddings.shape[1],
)
print(milvus_client.describe_collection(collection_name))


{'collection_name': 'hr_policy_manual', 'auto_id': False, 'num_shards': 1, 'description': '', 'fields': [{'field_id': 100, 'name': 'id', 'description': '', 'type': <DataType.INT64: 5>, 'params': {}, 'is_primary': True}, {'field_id': 101, 'name': 'vector', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 384}}], 'functions': [], 'aliases': [], 'collection_id': 468186072429387654, 'consistency_level': 2, 'consistency_level_name': 'Bounded', 'properties': {'timezone': 'UTC', 'namespace.sharding.enabled': 'false', 'max_field_id': '102'}, 'num_partitions': 1, 'enable_dynamic_field': True, 'enable_namespace': False, 'schema_version': 0, 'created_timestamp': 468187051387781138, 'update_timestamp': 468187051387781138}


In [39]:
data = [
    {
        "id": i,
        "vector": embeddings[i].tolist(),
        "text": chunk_texts[i],
        "page": chunk_metas[i]["page"],
        "headings": chunk_metas[i]["headings"],
    }
    for i in range(len(chunk_texts))
]
res = milvus_client.insert(collection_name=collection_name, data=data)
print(res)


{'insert_count': 500, 'ids': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 

In [40]:
milvus_client.flush(collection_name)

milvus_client.create_index(
    collection_name=collection_name,
    index_params=milvus_client.prepare_index_params(
        field_name="vector",
        index_type="AUTOINDEX",
        metric_type="COSINE",
    ),
)

milvus_client.load_collection(collection_name)
print(milvus_client.get_load_state(collection_name))


{'state': <LoadState: Loaded>}


In [41]:
def retrieve(query: str, top_k: int = 5):
    query_vec = embed_model.encode([query]).tolist()
    results = milvus_client.search(
        collection_name=collection_name,
        data=query_vec,
        limit=top_k,
        output_fields=["text", "page", "headings"],
    )
    return results[0]

hits = retrieve("How many days of casual leave are employees entitled to?", top_k=5)
for h in hits:
    print(f"score={h['distance']:.3f} page={h['entity']['page']} headings={h['entity']['headings']}")
    print(h['entity']['text'][:300])
    print("---")


score=0.697 page=79 headings=5.1 LEAVE TYPE 1: CASUAL LEAVE
5.1 LEAVE TYPE 1: CASUAL LEAVE
- 5.1.1 Casual leave admissible to an employee is eight days for a calendar year, subject to the condition that not more than five days' casual leave may be allowed at a time.
- 5.1.2 Casual leave can be combined with Special Casual leave but not with any other kind of
---
score=0.604 page=80 headings=5.2 LEAVE TYPE 2: EARNED LEAVE
5.2 LEAVE TYPE 2: EARNED LEAVE
- 5.2.1 The administrative staff is entitled to 30 days of Earned Leave.  Fifteen days will be credited to the employee's account on 1st of January and 15 days on 1st of July.
- 5.2.2 The existing ceiling on the accumulation of EL is 300 days.
- 5.2.3 The credit rate i
---
score=0.579 page=78 headings=(2) PROCEDURE FOR GRANTING LEAVE
(2) PROCEDURE FOR GRANTING LEAVE
- 2.1  The grant of leave to the Institute employee is governed by the Institute Leave Rules.  These rules are framed in line with the leave rules applicable to the Central Go

In [43]:
import os
from openai import OpenAI

# set in shell before starting jupyter: export OPENROUTER_API_KEY="your-key"
llm_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

def answer(query: str, top_k: int = 5, model: str = "anthropic/claude-sonnet-4.5", max_tokens: int = 500):
    hits = retrieve(query, top_k=top_k)
    context = "\n\n---\n\n".join(
        f"[page {h['entity']['page']} | {h['entity']['headings']}]\n{h['entity']['text']}"
        for h in hits
    )
    prompt = f"""Answer the question using ONLY the context below. Cite the page number(s) you used. If the answer isn't in the context, say so.

Context:
{context}

Question: {query}
"""
    response = llm_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content

print(answer("WHAT Qualification NEEDED FOR THE CFO POST ? "))

AuthenticationError: Error code: 401 - {'error': {'message': 'User not found.', 'code': 401}}

In [ ]:
!pip3 install langchain langchain-milvus langchain-huggingface langchain-openai

In [ ]:
from langchain_core.documents import Document

lc_documents = [
    Document(
        page_content=chunk_texts[i],
        metadata={"page": chunk_metas[i]["page"], "headings": chunk_metas[i]["headings"]},
    )
    for i in range(len(chunk_texts))
]

print(f"Total LangChain documents: {len(lc_documents)}")

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_milvus import Milvus

hf_embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = Milvus.from_documents(
    documents=lc_documents,
    embedding=hf_embeddings,
    collection_name="hr_policy_manual_lc",
    connection_args={"uri": "http://localhost:19530"},
)

print("Vectorstore ready")

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = ChatOpenAI(
    model="anthropic/claude-sonnet-4.5",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
    max_tokens=500,
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

prompt = ChatPromptTemplate.from_template(
    """Answer the question using ONLY the context below. Cite the page number(s) you used. If the answer isn't in the context, say so.

Context:
{context}

Question: {question}
"""
)

def format_docs(docs):
    return "\n\n---\n\n".join(
        f"[page {d.metadata.get('page')} | {d.metadata.get('headings')}]\n{d.page_content}"
        for d in docs
    )

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
print(rag_chain.invoke("How many days of casual leave are employees entitled to?"))